# Requests: talking to HTTP APIs

A local API contains records that must be retrieved completely and saved reliably. Treat every request as an uncertain operation: inspect status and representation, bound waiting and pagination, and verify that the exported records match the intended collection.

Requests sends HTTP requests and turns the response into a Python object. We will retrieve records, pass query parameters, submit JSON, handle errors, and download a file.

The examples use a tiny API running on your own computer. Its café menu is **invented teaching data**. No accounts, API keys, or internet connection are needed after installation. Run cells in order, including the final cleanup cell. The local server is teaching infrastructure, not a production web service.

## 1. Install and import

If needed, run `%pip install requests` in a separate cell.

| Concept | Meaning |
| --- | --- |
| URL | Address of a resource |
| Method | Action such as GET or POST |
| Headers | Metadata about the request or response |
| Body | Transmitted data, such as JSON or a file |
| Status code | Result of the HTTP operation |

GET typically retrieves a representation. POST typically submits data; its exact behavior is defined by the API.

In [ ]:
from pathlib import Path
import csv
import io
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlsplit, parse_qs

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("Requests:", requests.__version__)
TIMEOUT = (2, 5)  # Connection timeout, read timeout (seconds).

## 2. Start a local practice API

This setup cell creates endpoints for menu records, echoes, pagination, redirects, a download, and deliberately unsuccessful responses. It binds only to `127.0.0.1`, and the operating system selects an available port.

You can skim the server code on a first pass. The subsequent cells teach the Requests client. Rerunning this setup stops the previous practice server first.

In [ ]:
if "server" in globals():
    server.shutdown()
    server.server_close()
    server_thread.join(timeout=2)

MENU = [
    {"id": 1, "name": "Tea", "price": 3.0},
    {"id": 2, "name": "Coffee", "price": 4.0},
    {"id": 3, "name": "Sandwich", "price": 8.0},
    {"id": 4, "name": "Soup", "price": 6.0},
    {"id": 5, "name": "Cake", "price": 5.0},
]
DOWNLOAD_BYTES = b"id,name,price\n1,Tea,3.0\n2,Coffee,4.0\n"

class PracticeAPI(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def send_body(self, status, body, content_type="application/json; charset=utf-8", headers=None):
        self.send_response(status)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        for key, value in (headers or {}).items():
            self.send_header(key, value)
        self.end_headers()
        try:
            self.wfile.write(body)
        except (BrokenPipeError, ConnectionResetError):
            pass  # A client may deliberately time out in the timeout example.

    def send_json(self, status, payload, headers=None):
        self.send_body(status, json.dumps(payload).encode("utf-8"), headers=headers)

    def do_GET(self):
        parsed = urlsplit(self.path)
        query = parse_qs(parsed.query)
        if parsed.path == "/menu":
            term = query.get("q", [""])[0].lower()
            self.send_json(200, {"items": [item for item in MENU if term in item["name"].lower()]})
        elif parsed.path == "/echo":
            self.send_json(200, {"query": query, "client": self.headers.get("X-Demo-Client"),
                                 "cookie": self.headers.get("Cookie")})
        elif parsed.path == "/cookie":
            self.send_json(200, {"message": "Demo cookie set"}, {"Set-Cookie": "demo=visited; Path=/; SameSite=Lax"})
        elif parsed.path == "/pages":
            try:
                page = int(query.get("page", ["1"])[0])
            except ValueError:
                return self.send_json(400, {"error": "page must be an integer"})
            if page < 1 or page > 3:
                return self.send_json(400, {"error": "page must be between 1 and 3"})
            start = (page - 1) * 2
            self.send_json(200, {"items": MENU[start:start+2], "next_page": page+1 if page < 3 else None})
        elif parsed.path == "/redirect":
            self.send_body(302, b"", headers={"Location": "/menu"})
        elif parsed.path == "/download":
            self.send_body(200, DOWNLOAD_BYTES, "text/csv; charset=utf-8")
        elif parsed.path == "/text":
            self.send_body(200, "Hello from the café".encode("utf-8"), "text/plain; charset=utf-8")
        elif parsed.path == "/slow":
            time.sleep(0.3)
            self.send_json(200, {"message": "Delayed response"})
        elif parsed.path == "/flaky":
            self.server.flaky_calls += 1
            if self.server.flaky_calls == 1:
                self.send_json(503, {"error": "Try again"}, {"Retry-After": "0"})
            else:
                self.send_json(200, {"attempt": self.server.flaky_calls})
        else:
            self.send_json(404, {"error": "Resource not found"})

    def do_POST(self):
        if urlsplit(self.path).path != "/echo":
            return self.send_json(404, {"error": "Resource not found"})
        body = self.rfile.read(int(self.headers.get("Content-Length", "0")))
        content_type = self.headers.get("Content-Type", "")
        if content_type.startswith("application/json"):
            try:
                received = json.loads(body)
            except (ValueError, UnicodeDecodeError):
                return self.send_json(400, {"error": "Invalid JSON"})
        else:
            received = parse_qs(body.decode("utf-8"))
        self.send_json(200, {"received": received, "content_type": content_type})

server = ThreadingHTTPServer(("127.0.0.1", 0), PracticeAPI)
server.daemon_threads = True
server.flaky_calls = 0
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
BASE_URL = f"http://127.0.0.1:{server.server_port}"
print("Practice API:", BASE_URL)

## 3. Make a GET request

A Session stores shared settings and cookies and can reuse connections when the server supports it. This practice server closes each response connection, but the session behavior is still useful.

`trust_env=False` isolates these loopback examples from machine proxy and credential configuration. Ordinary external clients can keep the default. Pass a timeout on every call: Requests does not impose one automatically.

In [ ]:
if "client" in globals():
    client.close()
client = requests.Session()
client.trust_env = False
response = client.get(f"{BASE_URL}/menu", timeout=TIMEOUT)
response.raise_for_status()
menu_payload = response.json()
print("Status:", response.status_code)
print("URL:", response.url)
print("Content type:", response.headers["Content-Type"])
print("Items:", menu_payload["items"])

## 4. Send query parameters

Use `params` to encode query values instead of manually assembling a URL. Lists create repeated keys. Parameters are part of the URL; they are distinct from a POST body.

The server's echo lets us inspect how spaces, punctuation, and repeated values arrive.

In [ ]:
query_response = client.get(f"{BASE_URL}/echo", params={"q": "tea & cake", "tag": ["hot", "sweet"]}, timeout=TIMEOUT)
query_response.raise_for_status()
print("Encoded URL:", query_response.url)
print("Decoded parameters:", query_response.json()["query"])
filtered_response = client.get(f"{BASE_URL}/menu", params={"q": "tea"}, timeout=TIMEOUT)
filtered_response.raise_for_status()
print("Filtered items:", filtered_response.json()["items"])

## 5. Distinguish a response body from a usable record

Predict whether each response should be interpreted as bytes, text or JSON. Inspect the content type and status. A valid JSON body can still describe an API error instead of the records you wanted.

`content` provides bytes, `text` decodes them using an encoding, and `json()` parses JSON into Python objects. A response body is not necessarily JSON even when the status is 200.

Check HTTP status separately from parsing: an error response can contain valid JSON, while a successful response can contain plain text or an empty body.

In [ ]:
text_response = client.get(f"{BASE_URL}/text", timeout=TIMEOUT)
text_response.raise_for_status()
print("Bytes:", text_response.content)
print("Text:", text_response.text)
try:
    text_response.json()
except requests.exceptions.JSONDecodeError:
    print("Expected: this successful response contains text, not JSON.")

## 6. Share headers and cookies with a Session

A session applies common headers to subsequent requests. A response's `Set-Cookie` header stores a cookie, which the session sends back when its scope matches.

This example uses only a harmless demonstration cookie. The echo endpoint makes the outgoing metadata visible.

In [ ]:
client.headers.update({"X-Demo-Client": "requests-notebook", "Accept": "application/json"})
cookie_response = client.get(f"{BASE_URL}/cookie", timeout=TIMEOUT)
cookie_response.raise_for_status()
metadata_response = client.get(f"{BASE_URL}/echo", timeout=TIMEOUT)
metadata_response.raise_for_status()
print("Session cookies:", client.cookies.get_dict())
print("Server received:", metadata_response.json())

## 7. Submit JSON or form fields

`json=payload` serializes a Python object and supplies an application/json content type. `data=dict` encodes ordinary form fields. Choose the representation the API expects.

These POST requests only echo data; they do not create an order. A real API may create or modify a resource, so a repeated POST can have a repeated effect.

In [ ]:
order = {"item_id": 1, "quantity": 2}
json_response = client.post(f"{BASE_URL}/echo", json=order, timeout=TIMEOUT)
json_response.raise_for_status()
form_response = client.post(f"{BASE_URL}/echo", data={"item_id": "1", "quantity": "2"}, timeout=TIMEOUT)
form_response.raise_for_status()
print("JSON echo:", json_response.json())
print("Form echo:", form_response.json())

## 8. Do not turn a failed request into an empty dataset

Predict the status and exception path for each failure. Distinguish a server response reporting failure from a transport error with no response. Only a successful, validated result should enter the dataset.

4xx statuses describe client-side request problems; 5xx statuses describe server-side failures. `raise_for_status()` raises `HTTPError` for those ranges. It does not reject ordinary redirects.

Keep error handling specific so callers can decide what to do. Do not silently turn every failed request into an empty list, which could be mistaken for a successful response with no records.

In [ ]:
missing_status = None
try:
    missing_response = client.get(f"{BASE_URL}/missing", timeout=TIMEOUT)
    missing_response.raise_for_status()
except requests.exceptions.HTTPError as error:
    missing_status = error.response.status_code
    print("Expected HTTP error:", missing_status)
    print("Server explanation:", error.response.json())

## 9. Timeouts limit waiting

A tuple sets separate connection and read timeouts. The read timeout limits waiting for data, not the total duration of an entire download. Network problems can also raise `ConnectionError`; timeout and HTTP status failures are different cases.

Our slow endpoint delays its response by 0.3 seconds, longer than the 0.05-second read timeout below.

In [ ]:
timeout_observed = False
try:
    slow_response = client.get(f"{BASE_URL}/slow", timeout=(2, 0.05))
    slow_response.raise_for_status()
except requests.exceptions.Timeout:
    timeout_observed = True
    print("Expected: the server did not respond within the read timeout.")

## 10. Inspect redirects

GET requests follow redirects by default. `history` contains the intermediate responses, and `url` records the final destination. Disable automatic redirects when you want to inspect the initial response yourself.

In [ ]:
redirect_response = client.get(f"{BASE_URL}/redirect", timeout=TIMEOUT)
redirect_response.raise_for_status()
print("Redirect statuses:", [item.status_code for item in redirect_response.history])
print("Final URL:", redirect_response.url)
initial_redirect = client.get(f"{BASE_URL}/redirect", allow_redirects=False, timeout=TIMEOUT)
print("Initial status:", initial_redirect.status_code)
print("Location:", initial_redirect.headers["Location"])

## 11. Retry only when repeating the operation is appropriate

Inspect which methods and status codes the retry policy permits. Explain why repeating a GET differs from repeating an operation that creates a payment or record. Bound retries and waiting so a temporary issue does not become an endless run.

A bounded retry policy can help with a transient server failure. Here an adapter retries GET requests after selected server statuses. The endpoint fails once and then succeeds. Backoff reduces rapid repeated attempts; a server's Retry-After header can also affect waiting.

Retries are additional requests, not a repair for bad parameters or authentication. Do not automatically repeat a write operation without understanding the API's idempotency guarantees.

In [ ]:
retry_policy = Retry(total=2, backoff_factor=0.1, status_forcelist=[502, 503, 504],
                     allowed_methods=frozenset(["GET"]), respect_retry_after_header=True)
server.flaky_calls = 0
with requests.Session() as retry_client:
    retry_client.trust_env = False
    retry_client.mount("http://", HTTPAdapter(max_retries=retry_policy))
    retry_response = retry_client.get(f"{BASE_URL}/flaky", timeout=TIMEOUT)
    retry_response.raise_for_status()
    print("Successful result:", retry_response.json())
print("Server attempts:", server.flaky_calls)

## 12. Show that pagination reached the intended end

Predict the stopping condition and maximum visits. After collection, check record IDs, duplicates and total count. Receiving one successful page is not evidence that the complete dataset was retrieved.

This API returns two records per page and a `next_page` value. Other APIs may use cursor tokens or links instead; follow their documented contract.

Our helper checks the envelope, validates each record, rejects repeated pages, and limits page count. It keeps the server-provided page number on the same endpoint instead of following arbitrary URLs.

In [ ]:
def fetch_menu_pages(session, base_url, max_pages=10):
    records = []
    page = 1
    visited = set()
    for _ in range(max_pages):
        if type(page) is not int or page < 1 or page in visited:
            raise ValueError("Invalid or repeated page number")
        visited.add(page)
        result = session.get(f"{base_url}/pages", params={"page": page}, timeout=TIMEOUT)
        result.raise_for_status()
        payload = result.json()
        if not isinstance(payload, dict) or not isinstance(payload.get("items"), list) or "next_page" not in payload:
            raise ValueError("Unexpected page schema")
        for item in payload["items"]:
            if not isinstance(item, dict) or type(item.get("id")) is not int or not isinstance(item.get("name"), str):
                raise ValueError("Unexpected menu record")
            if type(item.get("price")) not in (int, float):
                raise ValueError("Price must be numeric")
        records.extend(payload["items"])
        page = payload["next_page"]
        if page is None:
            return records
    raise RuntimeError("Pagination exceeded max_pages")

all_items = fetch_menu_pages(client, BASE_URL)
print("Fetched records:", len(all_items))
print(all_items)

## 13. Stream a file to disk

`stream=True` allows incremental consumption through `iter_content`. A context manager closes the response, including when an error occurs. Write binary chunks to a file opened with `wb`.

We write a temporary partial file and rename it only after a complete download, so an interrupted transfer does not replace the previous completed file. This small sample demonstrates the mechanism; a real streaming download need not load the whole file into memory.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
download_path = output_dir / "requests_menu_download.csv"
partial_path = download_path.with_suffix(".csv.part")
try:
    with client.get(f"{BASE_URL}/download", stream=True, timeout=TIMEOUT) as download_response:
        download_response.raise_for_status()
        with partial_path.open("wb") as handle:
            for chunk in download_response.iter_content(chunk_size=16):
                if chunk:
                    handle.write(chunk)
    partial_path.replace(download_path)
finally:
    if partial_path.exists():
        partial_path.unlink()
print("Downloaded bytes:", download_path.stat().st_size)
print(download_path.read_text(encoding="utf-8"))

## 14. Complete exercise: export the paginated menu

Fetch all pages, reject duplicate IDs, and create a JSON snapshot plus a CSV table. The local POST echo did not change these records.

Running this cell replaces these named output files. The summary uses the fetched records and does not assume that a successful HTTP status guarantees the expected business data.

In [ ]:
export_items = fetch_menu_pages(client, BASE_URL)
ids = [item["id"] for item in export_items]
if len(ids) != len(set(ids)):
    raise ValueError("Duplicate item IDs across pages")
json_path = output_dir / "requests_menu.json"
csv_path = output_dir / "requests_menu.csv"
json_path.write_text(json.dumps(export_items, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "name", "price"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(export_items)
print(f"Exported {len(export_items)} menu records.")
print("Price range:", min(item["price"] for item in export_items), "to", max(item["price"] for item in export_items))
print("Saved:", json_path)
print("Saved:", csv_path)

## 15. Close the client and stop the server

Close the session and the practice server after finishing. To rerun request examples afterward, rerun the server setup and client setup cells first.

For an external API, substitute its documented URL, parameter names, and response schema. Keep HTTPS verification enabled, use its prescribed authentication method, and read credentials from configuration rather than writing real tokens into a notebook.

In [ ]:
client.close()
server.shutdown()
server.server_close()
server_thread.join(timeout=2)
print("Practice server stopped:", not server_thread.is_alive())

## 16. Practice and troubleshooting

1. Filter the menu for another word and inspect the encoded URL.
2. Set `max_pages=2` in the pagination helper and explain why it raises an error.
3. Add another field to the JSON POST and compare its type with a form-encoded value.
4. Change the downloaded chunk size and confirm that the final file bytes remain identical.

| Symptom | What to check |
| --- | --- |
| Connection refused | Start the local server and use its current port |
| Read timeout | Server speed and the read timeout; do not remove all timeout limits |
| JSON decoding fails | Status, content type, and whether the body is empty or non-JSON |
| HTTP 401 or 403 | The external API's authentication and authorization requirements |
| HTTP 429 | Rate limits and the API's documented retry guidance |
| Only the first page appears | Follow pagination metadata using a bounded loop |
| Download contains an error page | Check status before writing response bytes |
| A GET returns HTML but no browser-rendered content | Requests retrieves HTTP content; it does not execute page JavaScript |

## Further reading

The official [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) covers parameters, response bodies, and errors. See [advanced usage](https://requests.readthedocs.io/en/stable/user/advanced/) for sessions and streaming and the [API reference](https://requests.readthedocs.io/en/stable/api/) for method details.

## Explain what the evidence supports

Trace one exported record back to a page response. Explain how the client distinguishes a genuine empty result, a failed request and incomplete pagination.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: collect a paginated API under retries and malformed rows

An export endpoint returns overlapping pages, one transient 503, and one invalid record. Retrieve a consistent snapshot with bounded retries and pagination, quarantine the malformed row, and compare with the incomplete first-page approach. A local practice server reproduces the failures; it does not contact an external service.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through completeness before optimizing calls

A 200 response proves one request succeeded, not that the export is complete. If page 2 repeats the last record of page 1, concatenating without ID checks inflates counts. If the snapshot changes between pages, a deduplicated result can still omit records. Carry a snapshot identifier and verify it throughout the traversal.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from http.server import BaseHTTPRequestHandler,HTTPServer
from threading import Thread
from urllib.parse import urlparse,parse_qs
import json
p2_pages={1:{'snapshot':'v1','items':[{'id':i,'value':i*10} for i in [1,2,3]],'next':2},
 2:{'snapshot':'v1','items':[{'id':i,'value':i*10} for i in [3,4,5,6]],'next':3},
 3:{'snapshot':'v1','items':[{'id':7,'value':70},{'id':8,'value':'bad'}],'next':None}}
p2_attempts={};p2_calls=[]
class P2Handler(BaseHTTPRequestHandler):
    def log_message(self,*args):pass
    def do_GET(self):
        page=int(parse_qs(urlparse(self.path).query).get('page',['1'])[0]);p2_calls.append(page)
        p2_attempts[page]=p2_attempts.get(page,0)+1
        code=503 if page==2 and p2_attempts[page]==1 else 200
        body=json.dumps({'error':'temporary'} if code==503 else p2_pages[page]).encode()
        self.send_response(code);self.send_header('Content-Type','application/json');self.send_header('Content-Length',str(len(body)));self.end_headers();self.wfile.write(body)
p2_server=HTTPServer(('127.0.0.1',0),P2Handler);p2_thread=Thread(target=p2_server.serve_forever,daemon=True);p2_thread.start()
p2_url=f'http://127.0.0.1:{p2_server.server_port}/records'

### Work through a request and retry budget

Retries are allowed only for GET and limited to two additional attempts. Pagination is capped at five unique pages. These bounds prevent an endlessly repeating next link or failing server from creating an infinite loop. Keep the per-request timeout separate from the total-export budget; a Requests timeout is not a strict wall-clock deadline for the whole job.

In [ ]:
p2_seen={};p2_rejected=[];p2_visited=set();p2_duplicate_count=0
try:
    with requests.Session() as p2_session:
        p2_session.mount('http://',HTTPAdapter(max_retries=Retry(total=2,status_forcelist=[503],allowed_methods=['GET'],backoff_factor=0)))
        p2_first=p2_session.get(p2_url,params={'page':1},timeout=(2,5));p2_first.raise_for_status()
        p2_naive=len(p2_first.json()['items'])
        p2_snapshot=p2_first.json()['snapshot'];p2_page=1
        while p2_page is not None:
            if p2_page in p2_visited or len(p2_visited)>=5:raise ValueError('Pagination cycle or budget exceeded')
            p2_visited.add(p2_page)
            p2_response=p2_session.get(p2_url,params={'page':p2_page},timeout=(2,5));p2_response.raise_for_status()
            p2_payload=p2_response.json()
            if p2_payload['snapshot']!=p2_snapshot:raise ValueError('Snapshot changed during export')
            for p2_row in p2_payload['items']:
                if type(p2_row.get('id')) is not int or type(p2_row.get('value')) is not int:
                    p2_rejected.append(p2_row);continue
                if p2_row['id'] in p2_seen:
                    if p2_seen[p2_row['id']]!=p2_row:raise ValueError('Conflicting duplicate record')
                    p2_duplicate_count+=1
                else:p2_seen[p2_row['id']]=p2_row
            p2_page=p2_payload['next']
finally:
    p2_server.shutdown();p2_server.server_close();p2_thread.join(timeout=2)
assert set(p2_seen)==set(range(1,8)) and len(p2_rejected)==1 and p2_duplicate_count==1
assert p2_attempts[2]==2 and len(p2_visited)==3 and len(p2_calls)==5
pd.DataFrame(p2_seen.values()).to_csv(p2_out/'requests_export.csv',index=False)
print({'first_page_rows':p2_naive,'valid_unique_records':len(p2_seen),'rejected':len(p2_rejected),'actual_HTTP_calls_including_comparison':len(p2_calls)})

### Explain which failures were repaired

The transient GET failure was retried; the malformed record was quarantined rather than retried indefinitely; the repeated ID was reconciled; the first-page shortcut missed later records. Five HTTP calls include the initial comparison request, the three collector pages and one 503 retry. The server is closed even on failure.

**Try:** change page 3's snapshot to v2. **Worked reasoning:** the collector should stop instead of publishing a mixed snapshot. Restarting against a stable snapshot/cursor is safer than repeatedly continuing from a moving offset. Replaying a write request needs separate idempotency semantics; the GET retry policy should not be copied blindly to payments or creates.